# Kazhdan-Lusztig 方法使用演示

本 notebook 演示如何使用 `pyw` 库中的 Kazhdan-Lusztig 方法计算可容许表示的特征标。

## 目录

1. [数学背景](#1-数学背景)
2. [Bruhat 序](#2-bruhat-序)
3. [Kazhdan-Lusztig 多项式](#3-kazhdan-lusztig-多项式)
4. [形式特征标](#4-形式特征标)
5. [Verma 模特征标](#5-verma-模特征标)
6. [Kac-Wakimoto 特征标公式](#6-kac-wakimoto-特征标公式)
7. [完整示例：A₂⁽¹⁾ 在分数层次](#7-完整示例)

---

## 1. 数学背景

### Kazhdan-Lusztig 特征标公式

对于仿射 Kac-Moody 代数 $\mathfrak{g}$ 在可容许层次 $k$ 的不可约最高权模 $L_\lambda$，其特征标由 Kazhdan-Lusztig 公式给出：

$$
\text{ch}(L_\lambda) = \sum_{w \in W^{\text{aff}}} Q_{[w],[e]}(1) \cdot \text{ch}(M(w \cdot \Lambda))
$$

其中：
- $W^{\text{aff}}$ 是仿射 Weyl 群
- $Q_{[w],[e]}(1)$ 是逆 Kazhdan-Lusztig 多项式在 $q=1$ 处的值
- $M(w \cdot \Lambda)$ 是 Verma 模
- $w \cdot \Lambda$ 是点作用（dot action）

### 关键组件

1. **Bruhat 序**：Weyl 群上的偏序关系，决定哪些元素对需要计算 KL 多项式
2. **KL 多项式**：
   - 标准 KL 多项式 $P_{x,y}(q)$
   - 逆 KL 多项式 $\tilde{Q}_{x,y}(q)$，满足 $\sum_z P_{x,z}(q) \tilde{Q}_{z,y}(q) = \delta_{x,y}$
3. **Weyl-Kac 分母公式**：用于计算 Verma 模特征标
4. **形式特征标**：$q$-级数形式的特征标表示

---

## 环境设置

首先导入必要的模块：

In [2]:
# 导入 pyw 库

from pyw.core import AffineLieAlgebra, AffineWeight
from pyw.core.bruhat import BruhatOrder
from pyw.core.kazhdan_lusztig import KazhdanLusztigPolynomials
from pyw.core.character import FormalCharacter, WeylKacDenominator, VermaCharacter
from pyw.fractional import FractionalLevel

# 导入 SageMath
from sage.all import WeylGroup

print("✓ 模块导入成功")

✓ 模块导入成功


## 2. Bruhat 序

Bruhat 序是 Weyl 群上的一个偏序关系，对于 KL 多项式的计算至关重要。

### 定义

对于 Weyl 群 $W$ 中的元素 $x, y$，我们说 $x \leq y$（在 Bruhat 序下）当且仅当存在一个约化表达式：

$$y = s_{i_1} s_{i_2} \cdots s_{i_k}$$

使得 $x$ 可以通过删除其中某些 $s_{i_j}$ 得到。

### 示例：$A_2$ 型 Weyl 群

In [9]:
# 创建 A2 型 Weyl 群
W = WeylGroup(["A", 2])
bruhat = BruhatOrder(W)

# 获取基本元素
e = W.one()              # 单位元
s1 = W.simple_reflection(1)
s2 = W.simple_reflection(2)
w0 = W.long_element()    # 最长元

print("Weyl 群元素：")
print(f"  单位元 e: {e}")
print(f"  简单反射 s₁: {s1}")
print(f"  简单反射 s₂: {s2}")
print(f"  最长元 w₀: {w0}")
print(f"\n最长元的长度: ℓ(w₀) = {bruhat.length(w0)}")

Weyl 群元素：
  单位元 e: [1 0 0]
[0 1 0]
[0 0 1]
  简单反射 s₁: [0 1 0]
[1 0 0]
[0 0 1]
  简单反射 s₂: [1 0 0]
[0 0 1]
[0 1 0]
  最长元 w₀: [0 0 1]
[0 1 0]
[1 0 0]

最长元的长度: ℓ(w₀) = 3


In [10]:
# Bruhat 序比较
print("Bruhat 序比较：")
print(f"  e ≤ s₁: {bruhat.le(e, s1)}")
print(f"  e ≤ s₂: {bruhat.le(e, s2)}")
print(f"  s₁ ≤ s₁s₂: {bruhat.le(s1, s1 * s2)}")
print(f"  s₁ ≤ w₀: {bruhat.le(s1, w0)}")
print(f"  s₂ ≤ s₁: {bruhat.le(s2, s1)}  (不可比)")

Bruhat 序比较：
  e ≤ s₁: True
  e ≤ s₂: True
  s₁ ≤ s₁s₂: True
  s₁ ≤ w₀: True
  s₂ ≤ s₁: False  (不可比)


In [11]:
# Bruhat 区间 [e, w₀]
print("\nBruhat 区间 [e, w₀] 中的所有元素：")
interval = bruhat.interval(e, w0)
for w in interval:
    print(f"  ℓ={bruhat.length(w):2d}: {w}")

print(f"\n区间大小: |[e, w₀]| = {len(interval)}")


Bruhat 区间 [e, w₀] 中的所有元素：
  ℓ= 3: [0 0 1]
[0 1 0]
[1 0 0]
  ℓ= 2: [0 1 0]
[0 0 1]
[1 0 0]
  ℓ= 2: [0 0 1]
[1 0 0]
[0 1 0]
  ℓ= 1: [0 1 0]
[1 0 0]
[0 0 1]
  ℓ= 1: [1 0 0]
[0 0 1]
[0 1 0]
  ℓ= 0: [1 0 0]
[0 1 0]
[0 0 1]

区间大小: |[e, w₀]| = 6


## 3. Kazhdan-Lusztig 多项式

### 标准 KL 多项式 $P_{x,y}(q)$

对于 $x \leq y$ (Bruhat 序)，KL 多项式 $P_{x,y}(q)$ 满足：

- $P_{x,x}(q) = 1$
- $\deg(P_{x,y}) \leq \frac{\ell(y) - \ell(x) - 1}{2}$

### 逆 KL 多项式 $\tilde{Q}_{x,y}(q)$

逆 KL 多项式满足：

$$\sum_{z} P_{x,z}(q) \tilde{Q}_{z,y}(q) = \delta_{x,y}$$

在特征标公式中，我们需要 $\tilde{Q}_{x,y}(1)$（在 $q=1$ 处的值）。

In [12]:
# 创建 KL 多项式计算器
W = WeylGroup(["A", 2])
kl = KazhdanLusztigPolynomials(W)

e = W.one()
s1 = W.simple_reflection(1)
s2 = W.simple_reflection(2)
w0 = W.long_element()

print("标准 KL 多项式 P_{x,y}(1)：")
print(f"  P_{{e,e}}(1) = {kl.P(e, e, at_one=True)}")
print(f"  P_{{e,s₁}}(1) = {kl.P(e, s1, at_one=True)}")
print(f"  P_{{e,s₁s₂}}(1) = {kl.P(e, s1*s2, at_one=True)}")
print(f"  P_{{e,w₀}}(1) = {kl.P(e, w0, at_one=True)}")

标准 KL 多项式 P_{x,y}(1)：
  P_{e,e}(1) = 1
  P_{e,s₁}(1) = 1
  P_{e,s₁s₂}(1) = 1
  P_{e,w₀}(1) = 1


In [13]:
print("\n逆 KL 多项式 Q̃_{x,y}(1)：")
print(f"  Q̃_{{e,e}}(1) = {kl.Q_tilde(e, e, at_one=True)}")
print(f"  Q̃_{{e,s₁}}(1) = {kl.Q_tilde(e, s1, at_one=True)}")
print(f"  Q̃_{{s₁,w₀}}(1) = {kl.Q_tilde(s1, w0, at_one=True)}")

print("\n验证逆关系 Σ_z P_{e,z} Q̃_{z,w₀} = δ_{e,w₀}：")
interval = BruhatOrder(W).interval(e, w0)
total = sum(kl.P(e, z, at_one=True) * kl.Q_tilde(z, w0, at_one=True) for z in interval)
print(f"  结果 = {total} (应该等于 {1 if e == w0 else 0})")


逆 KL 多项式 Q̃_{x,y}(1)：
  Q̃_{e,e}(1) = 1
  Q̃_{e,s₁}(1) = -1
  Q̃_{s₁,w₀}(1) = 1

验证逆关系 Σ_z P_{e,z} Q̃_{z,w₀} = δ_{e,w₀}：
  结果 = 0 (应该等于 0)


## 4. 形式特征标

形式特征标是 $q$-级数形式的表示：

$$\text{ch} = \sum_{n=0}^{\infty} c_n q^n$$

其中 $c_n$ 是权重空间的维数，$q$ 是形式变量（对应能量或共形维数）。

In [14]:
# 创建形式特征标
ch1 = FormalCharacter({0: 1, 1: 2, 2: 3, 3: 5}, max_grade=10)
ch2 = FormalCharacter({0: 1, 1: 1, 2: 1}, max_grade=10)

print("特征标 1:")
print(f"  {ch1}")
print("\n特征标 2:")
print(f"  {ch2}")

print("\n特征标运算：")
print(f"  ch1 + ch2 = {ch1 + ch2}")
print(f"  2 * ch1 = {2 * ch1}")
print(f"  ch1.shift(1) = {ch1.shift(1)}  (乘以 q)")

特征标 1:
  FormalCharacter(1 + 2*q + 3*q^2 + 5*q^3)

特征标 2:
  FormalCharacter(1 + 1*q + 1*q^2)

特征标运算：
  ch1 + ch2 = FormalCharacter(2 + 3*q + 4*q^2 + 5*q^3)
  2 * ch1 = FormalCharacter(2 + 4*q + 6*q^2 + 10*q^3)
  ch1.shift(1) = FormalCharacter(1*q + 2*q^2 + 3*q^3 + 5*q^4)  (乘以 q)


## 5. Verma 模特征标

Verma 模 $M(\lambda)$ 的特征标由 Weyl-Kac 分母公式给出：

$$\text{ch}(M(\lambda)) = \frac{e^\lambda}{\prod_{\alpha > 0} (1 - e^{-\alpha})^{\text{mult}(\alpha)}}$$

对于仿射代数，这是一个 $q$-级数。

In [15]:
# 计算 A₂⁽¹⁾ 的 Verma 模特征标
ala = AffineLieAlgebra(["A", 2, 1])
Lambda = ala.fundamental_weights()

print(f"代数: {ala._cartan_type}")
print(f"基本权: Λ₀, Λ₁, Λ₂")

# 计算 M(Λ₁) 的特征标
Lambda1 = Lambda[1]
print(f"\n计算 ch(M(Λ₁)) 到 grade 5...")

verma = VermaCharacter(ala, Lambda1)
ch = verma.character(max_grade=5)

print("\n特征标展开：")
for grade, coeff in ch:
    if coeff != 0:
        print(f"  q^{grade}: {coeff}")

代数: ['A', 2, 1]
基本权: Λ₀, Λ₁, Λ₂

计算 ch(M(Λ₁)) 到 grade 5...

特征标展开：
  q^0: 1
  q^1: 3
  q^2: 9
  q^3: 22
  q^4: 51
  q^5: 108


## 6. Kac-Wakimoto 特征标公式

对于可容许层次 $k$ 的不可约最高权模 $L_\lambda$，Kac-Wakimoto 公式给出：

$$\text{ch}(L_\lambda) = \sum_{w \in W^{\text{aff}}/W_0} \tilde{Q}_{[w],[e]}(1) \cdot \text{ch}(M(w \cdot \Lambda))$$

其中：
- $W^{\text{aff}}/W_0$ 是仿射 Weyl 群模有限 Weyl 群的陪集
- $\tilde{Q}_{[w],[e]}(1)$ 是抛物逆 KL 多项式
- $w \cdot \Lambda = w(\Lambda + \rho) - \rho$ 是点作用

### 可容许层次

层次 $k$ 是可容许的，如果它可以写成：

$$k = -h^\vee + \frac{p}{u}$$

其中 $h^\vee$ 是对偶 Coxeter 数，$p, u$ 是正整数且 $\gcd(p, u) = 1$。

## 7. 完整示例：$A_2^{(1)}$ 在分数层次

我们计算 $A_2^{(1)}$ 在层次 $k = -3 + \frac{4}{3} = -\frac{5}{3}$ 的可容许表示特征标。

### 步骤概览

1. 创建仿射 Lie 代数和分数层次
2. 选择可容许权重
3. 使用 Kac-Wakimoto 公式计算特征标
4. 展开到指定 grade

In [42]:
# 步骤 1: 创建代数和层次
ala = AffineLieAlgebra(["A", 1, 1])
level = FractionalLevel(["A", 1, 1], p=2, u=3)


In [50]:
# 步骤 2: 选择权重
Lambda = ala.fundamental_weights()
Lambda1 = -4/3 * Lambda[1]


# 验证可容许性
from pyw.fractional import AdmissibleWeight
admissible = AdmissibleWeight(["A", 1, 1], Lambda1, level)
print(f"\n可容许性检查: {admissible.is_admissible()}")


可容许性检查: False


In [5]:
# 步骤 3 & 4: 计算特征标
print("\n" + "=" * 60)
print("计算 ch(L_{Λ₁}) 使用 Kac-Wakimoto 公式")
print("=" * 60)

try:
    kw = KacWakimotoCharacter(ala, level)
    ch = kw.character(Lambda1, max_grade=3, verbose=True)
    
    print("\n特征标展开：")
    for grade, coeff in ch:
        if coeff != 0:
            print(f"  q^{grade}: {coeff}")
    
    print("\n✓ 计算成功！")
    
except Exception as e:
    print(f"\n注意: 完整计算需要 coxeter3 后端")
    print(f"错误信息: {e}")
    print("\n提示: 安装 coxeter3 以获得更快的计算速度")


计算 ch(L_{Λ₁}) 使用 Kac-Wakimoto 公式
Computing character for weight: (Lambda[1]; 1; 0)
Dominant weight in orbit: (Lambda[1]; 1; 0)
w such that w·Λ = λ: [1 0 0]
[0 1 0]
[0 0 1]
W_Λ generators: {0, 1, 2}
W^0_Λ generators: set()
  Coset [[1 0 0]
[0 1 0]
[0 0 1]]: Q̃ = 1, w'·Λ = (Lambda[1]; 1; 0)
  Coset [[-1  1  1]
[ 0  1  0]
[ 0  0  1]]: Q̃ = 1, w'·Λ = (2*Lambda[1] + Lambda[2]; 1; -1)
  Coset [[ 1  0  0]
[ 1 -1  1]
[ 0  0  1]]: Q̃ = 1, w'·Λ = (-3*Lambda[1] + 2*Lambda[2]; 1; 0)
  Coset [[ 1  0  0]
[ 0  1  0]
[ 1  1 -1]]: Q̃ = 1, w'·Λ = (2*Lambda[1] - 2*Lambda[2]; 1; 0)
  Coset [[ 0 -1  2]
[ 1 -1  1]
[ 0  0  1]]: Q̃ = 1, w'·Λ = (-4*Lambda[1] + 4*Lambda[2]; 1; -1)
  Coset [[ 0  2 -1]
[ 0  1  0]
[ 1  1 -1]]: Q̃ = 1, w'·Λ = (4*Lambda[1] - 3*Lambda[2]; 1; -1)
  Coset [[-1  1  1]
[-1  0  2]
[ 0  0  1]]: Q̃ = 1, w'·Λ = (5*Lambda[2]; 1; -3)
  Coset [[ 1  0  0]
[ 2  0 -1]
[ 1  1 -1]]: Q̃ = 1, w'·Λ = (-4*Lambda[2]; 1; 0)
  Coset [[-1  1  1]
[ 0  1  0]
[-1  2  0]]: Q̃ = 1, w'·Λ = (4*Lambda[1]; 1; -2)
 

## 总结

本 notebook 演示了 Kazhdan-Lusztig 方法的完整流程：

1. **Bruhat 序** - Weyl 群上的偏序关系
2. **KL 多项式** - 标准和逆 KL 多项式的计算
3. **形式特征标** - $q$-级数表示和运算
4. **Verma 模** - 使用 Weyl-Kac 分母公式
5. **Kac-Wakimoto 公式** - 可容许表示的特征标

### 进一步阅读

- Kac, V. G., Wakimoto, M. "On rationality of W-algebras"
- Kazhdan, D., Lusztig, G. "Representations of Coxeter groups and Hecke algebras"
- Kac, V. G. "Infinite-dimensional Lie algebras" (3rd ed.)

### 相关示例

- `examples/kazhdan_lusztig_demo.py` - Python 脚本版本
- `demos/admissible.ipynb` - 可容许权重详细演示
- `demos/affine_lie_algebra_demo.ipynb` - 仿射 Lie 代数基础

---

**pyw 库文档**: https://github.com/panyw5/pyw